# **Argovis Argo Data Fetching & Processing**
This notebook helps you fetch, process, and store Argo float profile data from the Argovis API (https://argovis.colorado.edu/) for any region and time range.

## **Overview**

This is a reusable system to:
- Fetch JSON data from the Argovis Argo API

- Convert the complex JSON into a clean, flat Pandas DataFrame

- Save the data as .parquet or .csv for further analysis

In [1]:
import requests
import pandas as pd

## **SetUp Functions**

### **1. Fetch Profiles**
#### Inputs:
A dictionary of query params specifying:

- **startDate, endDate:** _In ISO 8601 timestamps_

- **polygon:** _Coordinates bounding the region_

- **data:** _Variables to fetch (e.g., temperature, salinity, pressure)_

#### Output:
JSON response from the API containing float profiles

In [ ]:
def fetch_argovis_profiles(params):
    url = "https://argovis-api.colorado.edu/argo"
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()

### **2. JSON to DF**
#### Inputs:
JSON list of profiles from Argovis API

#### Output:
Cleaned Pandas DataFrame with:

- Metadata (platform number, timestamp, location, cycle number, etc.)

- Measurements (temperature, salinity, pressure, etc.)

#### Auto-normalization of columns:
Column names like PRESSURE, Pres, etc., are converted to lowercase standardized names (pres, temp, psal).

In [3]:
"""
    Convert Argovis JSON (list of profiles or single profile) into a flattened DataFrame.
    Handles both:
      - profile['measurements'] (list of dicts per level)
      - profile['data'] + profile['data_info'] (columns described in data_info[0])
"""

def argovis_json_to_df(profiles):
    if not isinstance(profiles, list):
        profiles = [profiles]

    rows = []
    for p in profiles:
        # --- metadata common to each level
        meta = {
            "platform_number": (p.get("_id") or "").split("_")[0] if p.get("_id") else None,
            "date": p.get("timestamp"),
            "date_qc": p.get("timestamp_argoqc"),
            "cycle_number": p.get("cycle_number"),
            "direction": p.get("profile_direction"),
            "position_qc": p.get("geolocation_argoqc"),
        }

        # geolocation -> lon/lat
        geoloc = p.get("geolocation")
        if isinstance(geoloc, dict):
            coords = geoloc.get("coordinates")
            if isinstance(coords, (list, tuple)) and len(coords) >= 2:
                meta["lon"] = coords[0]
                meta["lat"] = coords[1]

        # --- Case A: 'measurements' key (list of dicts)
        if "measurements" in p and p["measurements"]:
            for lvl in p["measurements"]:
                row = dict(meta)
                # lvl is typically a dict like {'pres':..., 'temp':..., 'psal':...}
                row.update(lvl)
                rows.append(row)
            continue

        # --- Case B: 'data' + 'data_info' layout
        if "data" in p and p.get("data") and "data_info" in p and p.get("data_info"):
            var_names = p["data_info"][0]  # list of variable names (e.g. ["pressure","temperature","salinity"...])
            data_block = p["data"]

            # data_block could be:
            #  - a list-of-lists whose order matches var_names (common)
            #  - a dict mapping varname -> list_of_values (less common)
            # Determine number of levels using pressure when available, else first variable
            def safe_len_from(idx):
                try:
                    return len(data_block[idx])
                except Exception:
                    return None

            n_levels = None

            # try pressure first (common name 'pressure' or 'pres' or 'PRES')
            for pres_name in ("pressure", "pres", "PRES", "Pressure"):
                if pres_name in var_names:
                    idx = var_names.index(pres_name)
                    n_levels = safe_len_from(idx)
                    break
            if n_levels is None:
                # fallback: first variable
                if isinstance(data_block, list) and len(data_block) > 0:
                    n_levels = safe_len_from(0)
                elif isinstance(data_block, dict):
                    # pick any value list
                    any_vals = next(iter(data_block.values()))
                    n_levels = len(any_vals) if hasattr(any_vals, "__len__") else 0
                else:
                    n_levels = 0

            # Build rows by zipping across variables
            for i in range(n_levels):
                row = dict(meta)
                for vi, vname in enumerate(var_names):
                    val = None
                    # try list-of-lists style
                    try:
                        if isinstance(data_block, list):
                            val = data_block[vi][i]
                        elif isinstance(data_block, dict):
                            # some endpoints put data as dict var->list
                            val = data_block.get(vname, [None]*n_levels)[i]
                    except Exception:
                        val = None
                    # store with a clean column name (keep varname as-is too)
                    row[vname] = val
                rows.append(row)
            continue

        # --- no measurement data for this profile (skip)
        continue

    df = pd.DataFrame(rows)

    # normalize some common column names if present (optional)
    # e.g., make PRES, pressure, etc. consistently named:
    
    col_map = {}
    for c in df.columns:
        if c.upper() in ("PRES", "PRESSURE"):
            col_map[c] = "pres"
        elif c.upper() in ("TEMP", "TEMPERATURE"):
            col_map[c] = "temp"
        elif c.upper() in ("PSAL", "SALINITY", "PSALINITY"):
            col_map[c] = "psal"
    if col_map:
        df = df.rename(columns=col_map)

    # convert date column to datetime if present
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    return df

## **Example Usage**

### **1. Define Parameters**

<u>Example 1</u>: _Fetch Profiles in a Small Region (2015)_

In [ ]:
params_test = {
    "startDate": "2015-01-01T00:00:00Z",
    "endDate":   "2015-12-31T23:59:59Z",
    # polygon as a JSON-like string; requests will urlencode it correctly
    "polygon": "[[-95,-3],[-90,-3],[-90,2],[-95,2],[-95,-3]]",

    # IMPORTANT: ask for variables using the `data` parameter.
    # Example format: "temperature,1,salinity,1,pressure,1"
    # (the '1' after each variable means: include that variable with QC flag 1).
    "data": "temperature,1,salinity,1,pressure,1"
}

<u>Example 2</u>: _Indian Ocean Profiles (2025-Jan)_

In [9]:
params_indian_ocean = {
    'startDate': '2025-01-01T00:00:00Z',
    'endDate': '2025-01-31T23:59:59Z',
    'polygon': "[[20,-40],[120,-40],[120,30],[20,30],[20,-40]]",
    'data': 'temperature,1,salinity,1,pressure,1'
}

## **Fetch & Convert Data**

In [ ]:
# Fetch data from Argovis
json_profiles = fetch_argovis_profiles(params_test)

# Convert to DataFrame
df = argovis_json_to_df(json_profiles)

df.head()

,platform_number,date,date_qc,cycle_number,direction,position_qc,lon,lat,temp,psal,pres
0,3901178,2015-04-23 04:34:29+00:00,1,32,A,1,-94.009,1.921,28.785999,33.466000,4.17
1,3901178,2015-04-23 04:34:29+00:00,1,32,A,1,-94.009,1.921,28.802000,33.507999,6.07
2,3901178,2015-04-23 04:34:29+00:00,1,32,A,1,-94.009,1.921,28.806999,33.528999,8.07
3,3901178,2015-04-23 04:34:29+00:00,1,32,A,1,-94.009,1.921,28.802000,33.569000,10.07
4,3901178,2015-04-23 04:34:29+00:00,1,32,A,1,-94.009,1.921,28.778999,33.630001,12.07


In [6]:
df.shape

(189941, 11)

In [7]:
df['platform_number'].nunique()

16

In [8]:
list(df['platform_number'].unique())

['3901178',
 '5903865',
 '3901154',
 '3901152',
 '3901171',
 '3901176',
 '3901151',
 '3900674',
 '3901164',
 '3900784',
 '5902247',
 '3901149',
 '3901150',
 '3901182',
 '5903448',
 '3901115']

#### Data fetching for Indian Ocean 

In [10]:
json_prof_IO = fetch_argovis_profiles(params_indian_ocean)

df = argovis_json_to_df(json_prof_IO)

df.head()

,platform_number,date,date_qc,cycle_number,direction,position_qc,lon,lat,temp,psal,pres
0,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.934999,33.874001,4.48
1,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,6.28
2,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,8.28
3,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,10.28
4,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.938000,33.875000,12.28


In [11]:
df.shape

(1054757, 11)

In [12]:
df['platform_number'].nunique()

481

## **Save Data for Future Use**

#### 1. Save to parquet

In [14]:
df.to_parquet('indian_ocean_profiles_01Jan2025_31Jan2025.parquet', index=False)

#### 2. Save to CSV (_Not Recommended_)

In [18]:
df.to_csv('io_sample1.csv')

## **Reload Parquet File**

In [15]:
pf = pd.read_parquet('indian_ocean_profiles_01Jan2025_31Jan2025.parquet')

In [ ]:
pf.head()

,platform_number,date,date_qc,cycle_number,direction,position_qc,lon,lat,temp,psal,pres
0,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.934999,33.874001,4.48
1,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,6.28
2,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,8.28
3,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.937000,33.875000,10.28
4,5905521,2025-01-31 23:56:21+00:00,1,99,A,1,107.0376,-8.2858,28.938000,33.875000,12.28


## **Notes & Recommendations**

- Use .parquet for faster loading, especially with large datasets.

- Make sure your params['polygon'] is correctly formatted as a list of coordinate pairs.

- Keep an eye on API timeout settings if you fetch large amounts of data (timeout=60 by default).

- Not all profiles contain measurement data, so always check for empty DataFrames.

## **Conclusion**
- This notebook enables easy and consistent extraction of Argo float profiles from Argovis API and prepares data for analysis or modeling workflows.
<br><br>
👉 Feel free to customize params as needed for your region, timeframe, and variable requirements.
----

## **HOW TO USE THE STORED DATASET**
The following section explains how to load, explore, filter,
and visualize the dataset that we saved as a Parquet file.
This ensures you don’t need to call the Argovis API again.

### **1. Load the Parquet File**

#### Import pandas

In [21]:
# (Already done before)
# import pandas as pd

#### Load the dataset into a DF
_NOTE: Make sure the parquet file is in the same folder as this notebook_

In [ ]:
df = pd.read_parquet("indian_ocean_profiles_01Jan2025_31Jan2025.parquet")

#### Basic Analysis

In [ ]:
# Show the shape (rows, columns)
print("DataFrame shape:", df.shape)

# Preview the first 5 rows
df.head()

### **2. Explore Metadata**

In [ ]:
# Count how many unique floats (platforms) are present
print("Unique floats in this dataset:", df['platform_number'].nunique())

# List all float IDs
print("List of floats:", df['platform_number'].unique())

# Count number of cycles recorded per float
print(df.groupby("platform_number")["cycle_number"].nunique())

### **3. Filter specific subsets of data**

#### Example 1: _Extract data for a specific float and cycle_

In [ ]:
subset = df[(df["platform_number"] == "3901178") & (df["cycle_number"] == 32)]

print("Subset shape (float 3901178, cycle 32):", subset.shape)

subset.head()

#### Example 2: _Extract only temperature-related columns_

In [ ]:
temp_profiles = df[["date", "lat", "lon", "pres", "temp"]]

print("Temperature-only DataFrame shape:", temp_profiles.shape)

### **4. Visualization examples**

#### Import pyplot from matplotlib

In [ ]:
import matplotlib.pyplot as plt

#### (A) Temperature-Salinity (T-S) diagram

In [ ]:
plt.scatter(df["psal"], df["temp"], s=5, alpha=0.5)
plt.xlabel("Salinity (psu)")
plt.ylabel("Temperature (°C)")
plt.title("Indian Ocean (Jan 2025) - T-S Diagram")
plt.show()

#### (B) Temperature vs Pressure profile for a float

In [ ]:
float_id = "3901178"
cycle = 32
sub = df[(df["platform_number"] == float_id) & (df["cycle_number"] == cycle)]